# ESM-2 embedding cache — protease cleavage model

Takes `training_pairs.csv` (194,182 rows) and produces one fixed-size numeric
vector per row: the P4-P4' window, each residue as an ESM-2 embedding.

**Run this once.** Afterwards training is fast and never touches ESM-2 again.

### Before you start
- Runtime -> Change runtime type -> **GPU** (T4 is enough)
- Have `training_pairs.csv` and `combined_clean.csv` ready to upload

### What comes out
`windows_<model>.npy`  — shape (194182, 8, D), float16, aligned row-for-row with the CSV
`cache_meta.json`      — model name, dims, row count, checksum

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip -q install fair-esm==2.0.0

In [ ]:
import torch, esm, numpy as np, pandas as pd, json, hashlib, os, time
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "Switch runtime to GPU: Runtime -> Change runtime type -> GPU"
print(torch.cuda.get_device_name(0))

## 2. Choose model size

`t33_650M` is the standard choice and what I'd use for the final run.
`t12_35M` is 18x smaller — use it for a first end-to-end pass to confirm the
pipeline works before spending the GPU time.

| Model | Dim | Output size | T4 time (3,355 proteins) |
|---|---|---|---|
| `esm2_t12_35M_UR50D`  | 480  | 1.5 GB | ~8 min |
| `esm2_t30_150M_UR50D` | 640  | 2.0 GB | ~20 min |
| `esm2_t33_650M_UR50D` | 1280 | 4.0 GB | ~60 min |

In [ ]:
MODEL_NAME = "esm2_t33_650M_UR50D"   # <- switch to esm2_t12_35M_UR50D for a fast trial run
W          = 4                        # P4-P4', 8 residues total
MAX_LEN    = 1022                     # ESM-2 position limit (1024 minus BOS/EOS)
BATCH_AA   = 12000                    # residues per batch; lower if you hit OOM

model, alphabet = esm.pretrained.__dict__[MODEL_NAME]()
model = model.eval().half().to(DEVICE)
REPR_LAYER = model.num_layers
DIM        = model.embed_dim
bc         = alphabet.get_batch_converter()
print(f"{MODEL_NAME}: {model.num_layers} layers, dim {DIM}")

## 3. Upload data

In [ ]:
from google.colab import files
up = files.upload()          # select training_pairs.csv AND combined_clean.csv
pairs = pd.read_csv("training_pairs.csv")
clean = pd.read_csv("combined_clean.csv")
seqs  = clean.drop_duplicates("substrate_uniprot").set_index("substrate_uniprot").substrate_sequence.to_dict()

print(f"{len(pairs):,} rows, {pairs.substrate_uniprot.nunique():,} proteins")
missing = set(pairs.substrate_uniprot) - set(seqs)
assert not missing, f"{len(missing)} proteins have no sequence"
assert pairs.label.isin([0,1]).all()

## 4. Long-protein handling

462 of your proteins are longer than 1,022 residues (max 8,797). Truncating
would silently drop their cleavage sites, so instead each needed position gets
a **crop centred on it**. The window residues always sit in the middle of the
crop with maximal real context on both sides, which is what ESM-2 needs to
produce a meaningful contextual embedding.

Short proteins pass through whole and are embedded once, shared across all their
rows. Long proteins get one forward pass per crop, grouped so overlapping
positions reuse the same crop.

In [ ]:
def plan_crops(uid, positions):
    """Return [(crop_seq, crop_start, [positions covered])]. crop_start is 0-based
    offset of the crop within the full sequence."""
    s = seqs[uid]; L = len(s)
    if L <= MAX_LEN:
        return [(s, 0, sorted(positions))]
    out, remaining = [], sorted(positions)
    while remaining:
        p = remaining[0]
        start = max(0, min(p - MAX_LEN // 2, L - MAX_LEN))
        end   = start + MAX_LEN
        # every position whose full window fits inside this crop
        covered = [q for q in remaining if q - W >= start and q + W <= end]
        if not covered:                      # pathological; force-cover p
            covered = [p]
        out.append((s[start:end], start, covered))
        remaining = [q for q in remaining if q not in set(covered)]
    return out

# group rows by protein -> positions
need = pairs.groupby("substrate_uniprot").cleavage_position.apply(lambda x: sorted(set(x)))
jobs = []
for uid, pos in need.items():
    for crop, start, covered in plan_crops(uid, pos):
        jobs.append((uid, crop, start, covered))

n_long = sum(len(seqs[u]) > MAX_LEN for u in need.index)
print(f"{len(need):,} proteins -> {len(jobs):,} forward passes ({n_long} long proteins need >1 crop)")

## 5. Embed

Batched by total residue count rather than sequence count, so a batch of short
proteins and a batch of long ones use similar GPU memory. If you hit OOM, lower
`BATCH_AA`.

Results are written into a dict keyed by `(uniprot, position)`. Runs in fp16
throughout — the precision loss is irrelevant here and it halves both time and
memory.

In [ ]:
jobs.sort(key=lambda j: len(j[1]))          # length-sorted = less padding waste
store = {}
t0 = time.time()
i = 0

while i < len(jobs):
    batch, aa = [], 0
    while i < len(jobs) and (not batch or aa + len(jobs[i][1]) <= BATCH_AA):
        batch.append(jobs[i]); aa += len(jobs[i][1]); i += 1

    _, _, toks = bc([(f"s{k}", j[1]) for k, j in enumerate(batch)])
    with torch.no_grad():
        rep = model(toks.to(DEVICE), repr_layers=[REPR_LAYER])["representations"][REPR_LAYER]

    for k, (uid, crop, start, covered) in enumerate(batch):
        emb = rep[k, 1:len(crop)+1]                     # strip BOS; now 0-based == crop index
        for p in covered:
            lo = p - W - start                          # cleavage between p-1 and p (1-based)
            store[(uid, p)] = emb[lo:lo+2*W].to(torch.float16).cpu().numpy()

    if i % 2000 < len(batch):
        el = time.time() - t0
        print(f"{i:,}/{len(jobs):,}  {el:.0f}s elapsed  ~{el/max(i,1)*(len(jobs)-i):.0f}s left")

print(f"done in {time.time()-t0:.0f}s, {len(store):,} windows cached")

## 6. Assemble, verify, save

The array is built in the **exact row order of `training_pairs.csv`**, so
`windows[i]` corresponds to `pairs.iloc[i]`. Keep them together.

In [ ]:
windows = np.zeros((len(pairs), 2*W, DIM), dtype=np.float16)
bad = []
for i, r in enumerate(pairs.itertuples()):
    v = store.get((r.substrate_uniprot, r.cleavage_position))
    if v is None or v.shape != (2*W, DIM):
        bad.append(i); continue
    windows[i] = v

print(f"filled {len(pairs)-len(bad):,} / {len(pairs):,}   missing {len(bad)}")
if bad:
    pairs = pairs.drop(index=bad).reset_index(drop=True)
    windows = np.delete(windows, bad, axis=0)
    print(f"dropped {len(bad)} unfillable rows -> {len(pairs):,}")

In [ ]:
# --- verification: the embedded window must match the sequence letters ---
import random
rng = random.Random(0)
for i in rng.sample(range(len(pairs)), 200):
    r = pairs.iloc[i]
    s = seqs[r.substrate_uniprot]; p = int(r.cleavage_position)
    assert s[p-1] == r.P1_ann and s[p] == r.P1p_ann, f"row {i}: annotation mismatch"
    assert s[p-W:p+W] == r.window_P4_P4p,            f"row {i}: window mismatch"
    assert not np.allclose(windows[i], 0),           f"row {i}: all-zero embedding"
print("200/200 spot checks passed")

# embeddings of identical windows should differ (context-dependence sanity)
dup = pairs[pairs.duplicated('window_P4_P4p', keep=False)].head(2)
if len(dup) == 2:
    a, b = windows[dup.index[0]], windows[dup.index[1]]
    print(f"identical 8-mers in different proteins, mean abs diff: {np.abs(a-b).mean():.4f} (should be > 0)")

In [ ]:
tag = MODEL_NAME.replace("esm2_","").replace("_UR50D","")
np.save(f"windows_{tag}.npy", windows)
pairs.to_csv("training_pairs_aligned.csv", index=False)

meta = {
    "model": MODEL_NAME, "dim": int(DIM), "window": 2*W, "rows": int(len(pairs)),
    "positives": int(pairs.label.sum()),
    "tiers": pairs.tier.value_counts().to_dict(),
    "sha256_pairs": hashlib.sha256(open("training_pairs_aligned.csv","rb").read()).hexdigest()[:16],
}
json.dump(meta, open("cache_meta.json","w"), indent=2)
print(json.dumps(meta, indent=2))
print(f"windows_{tag}.npy  {os.path.getsize('windows_' + tag + '.npy')/1e9:.2f} GB")

## 7. Get the files out

Colab wipes local disk when the runtime ends. Mount Drive and copy — the 650M
output is ~4 GB, so check you have room.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
dest = "/content/drive/MyDrive/protease_cache"
os.makedirs(dest, exist_ok=True)
!cp windows_*.npy training_pairs_aligned.csv cache_meta.json {dest}/
!ls -lh {dest}

## 8. Load it back (next notebook)

```python
import numpy as np, pandas as pd, json
meta    = json.load(open("cache_meta.json"))
pairs   = pd.read_csv("training_pairs_aligned.csv")
windows = np.load(f"windows_{meta['model'].replace('esm2_','').replace('_UR50D','')}.npy")
assert len(pairs) == len(windows) == meta["rows"]

X = windows.reshape(len(pairs), -1).astype("float32")   # (N, 8*D) for LightGBM
y = pairs.label.values
```

Then: one-hot the 29 `merops_id` values, concatenate, cluster substrates at 30%
identity with MMseqs2, split by cluster, train LightGBM, report AUPRC, and run
the three ablations.